In [1]:
# Preparacion

import sys
sys.path.append("..")

import os
import sqlite3
import numpy as np
import pandas as pd

from src.data import cargar_enemdu, mediana_ponderada
from src.features import agregar_variables

df = agregar_variables(cargar_enemdu())
os.makedirs("../dashboard/data", exist_ok=True)

df["nacional"] = "Nacional"
df["nivel"] = df["nnivins"].map({1: "Ninguno", 3: "Básica", 4: "Media", 5: "Superior"})   # alfabetización excluida (n pequeño)
df["grupo_etnico"] = df["etnia"].map({"Indígena": "Indígena", "Montuvio": "Montuvio",
                                      "Afroecuatoriano": "Afrodescendiente", "Negro": "Afrodescendiente",
                                      "Mulato": "Afrodescendiente", "Blanco": "Blanco", "Mestizo": "Mestizo"})

In [2]:
# Indicadores en formato largo

def tabla_indicadores(d, dim, col):
    filas = []
    for cat, s in d.groupby(col, observed=True):
        c = s["condact"]
        pet, pea, oc = c.between(1, 9), c.between(1, 8), c.between(1, 6)
        defs = {
            "Participación global": (pea, pet),
            "Empleo adecuado": (c == 1, pea),
            "Subempleo": (c.isin([2, 3]), pea),
            "Desempleo": (c.isin([7, 8]), pea),
            "Informalidad": (s["secemp"] == 2, oc),
        }
        for ind, (num, den) in defs.items():
            filas.append({"dimension": dim, "categoria": str(cat), "indicador": ind,
                          "peso_num": s.loc[num, "fexp"].sum(),
                          "peso_den": s.loc[den, "fexp"].sum(),
                          "n_den": int(den.sum())})
    return pd.DataFrame(filas)

dims = {"Nacional": "nacional", "Sexo": "sexo", "Zona": "zona", "Grupo de edad": "grupo_edad",
        "Dominio": "dominio_nombre", "Nivel de instrucción": "nivel", "Etnia": "grupo_etnico"}
indic = pd.concat([tabla_indicadores(df, d, c) for d, c in dims.items()], ignore_index=True)

check = indic[indic["dimension"] == "Nacional"].copy()
check["valor"] = (100 * check["peso_num"] / check["peso_den"]).round(1)
print(check[["indicador", "valor", "n_den"]])

indic.to_csv("../dashboard/data/indicadores.csv", index=False, encoding="utf-8-sig")

              indicador  valor  n_den
0  Participación global   63.7  66501
1       Empleo adecuado   35.7  41680
2             Subempleo   18.4  41680
3             Desempleo    3.4  41680
4          Informalidad   53.5  39920


In [4]:
# Informalidad por rama

con = sqlite3.connect("../data/processed/enemdu.db")
dim_rama = pd.read_sql_query("SELECT codigo, nombre AS rama FROM dim_rama", con)

oc = df[df["condact"].between(1, 6)].copy()
oc["peso_inf"] = (oc["secemp"] == 2) * oc["fexp"]
rama = (oc.groupby("rama1").agg(peso_ocupados=("fexp", "sum"), peso_informales=("peso_inf", "sum"), n=("fexp", "size"))
          .reset_index().merge(dim_rama, left_on="rama1", right_on="codigo").drop(columns=["rama1", "codigo"]))
print(rama.sort_values("peso_informales", ascending=False).head(3))
rama.to_csv("../dashboard/data/rama.csv", index=False, encoding="utf-8-sig")

   peso_ocupados  peso_informales     n                 rama
0   2.689227e+06     2.314129e+06  9179  Agricultura y pesca
6   1.312271e+06     5.847108e+05  7395             Comercio
5   5.664511e+05     4.300055e+05  2439         Construcción


In [6]:
# Piramide e ingresos

pea = df[df["condact"].between(1, 8)].copy()
bordes = list(range(15, 80, 5)) + [120]
etiquetas = [f"{a}-{a+4}" for a in range(15, 75, 5)] + ["75+"]
pea["tramo"] = pd.cut(pea["p03"], bins=bordes, labels=etiquetas, right=False)
pir = (pea.groupby(["tramo", "sexo"], observed=True)["fexp"].sum().div(1000).round(1).reset_index(name="miles"))
pir["miles_pir"] = np.where(pir["sexo"] == "Hombre", -pir["miles"], pir["miles"])
print(pir.groupby("sexo")["miles"].sum())
pir.to_csv("../dashboard/data/piramide.csv", index=False, encoding="utf-8-sig")

ing = df[df["condact"].between(1, 6) & (df["ingrl"] > 0)]
filas = []
for dim, col in {"Sexo": "sexo", "Zona": "zona", "Nivel de instrucción": "nivel"}.items():
    for cat, s in ing.groupby(col):
        filas.append({"dimension": dim, "categoria": cat,
                      "mediana": mediana_ponderada(s["ingrl"], s["fexp"]), "n": len(s)})
ingresos = pd.DataFrame(filas)
print(ingresos)
ingresos.to_csv("../dashboard/data/ingresos.csv", index=False, encoding="utf-8-sig")

sexo
Hombre    5109.1
Mujer     3510.6
Name: miles, dtype: float64
              dimension categoria  mediana      n
0                  Sexo    Hombre    430.0  20685
1                  Sexo     Mujer    350.0  14617
2                  Zona     Rural    240.0   9046
3                  Zona    Urbana    480.0  26256
4  Nivel de instrucción    Básica    270.0  10905
5  Nivel de instrucción     Media    430.0  12699
6  Nivel de instrucción   Ninguno    155.0    618
7  Nivel de instrucción  Superior    663.0  11050
